# Day 02 미니 프로젝트 · 방대한 문서로 실전 RAG 시스템 구축
예시 몇 줄이 아니라 **여러 문서로 이루어진 지식베이스**를 직접 청킹하고, **메타데이터로 필터링**하며, 고급 검색과 평가를 거쳐 하나의 RAG 시스템으로 완성합니다.

분량은 약 2시간을 기준으로 구성했습니다.

### 진행
- Part 1 · 방대한 문서 확보 → 청킹 + 메타데이터 부착 → 벡터DB 저장
- Part 2 · 메타데이터 필터링 검색 (카테고리 · 부서 · 연도)
- Part 3 · 검색 기법 비교 + 평가 (Recall@k · MRR)
- Part 4 · 최종 RAG 시스템 (필터 + 하이브리드 + 리랭커 + 생성)

### 제출물
1. 청킹 결과와 메타데이터 스키마
2. 필터 있는/없는 검색 비교
3. 기법별 점수표
4. 내 문서로 바꾼 최종 시스템 + 회고
---

## 0 · 설치

In [ ]:
!pip install -q sentence-transformers qdrant-client langchain-text-splitters openai rank_bm25

## 1 · NVIDIA API 토큰 설정
생성 단계에서만 사용합니다. 검색·필터·평가는 전부 로컬입니다.

In [ ]:
import os, getpass
from openai import OpenAI
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
_c = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("엔드포인트:", LLM_BASE_URL, "| 모델:", LLM_MODEL)

## Part 1 · 방대한 문서 확보
### 2 · 지식베이스 생성
사내 지식베이스를 생성합니다. 정책 문서 몇 건과, 템플릿으로 만든 여러 제품 매뉴얼을 합칩니다.
각 문서에는 `doc_id · title · category · department · year` 메타데이터가 붙습니다.

In [ ]:
# 정책 문서 (카테고리별 상세 문서)
POLICY = [
 dict(title="연차 및 휴가 규정", category="인사", department="인사팀", year=2024, body=(
   "연차 휴가는 입사 1년 차부터 매년 15일이 부여되며 근속 연수에 따라 최대 25일까지 늘어난다. "
   "연차는 사용 3영업일 전까지 사내포털의 휴가 신청 메뉴에서 신청하고 팀장의 승인을 받아야 한다. "
   "반차와 반반차 단위 사용도 가능하며 동일한 절차를 따른다. 승인 없이 사용하면 결근으로 처리될 수 있다. "
   "연차는 회계연도 기준으로 이월되지 않으며 남은 연차는 12월 말까지 소진하는 것을 권장한다.")),
 dict(title="비용 정산 절차", category="재무", department="재무팀", year=2024, body=(
   "비용 정산은 영수증을 첨부해 신청서를 작성하고 경영지원팀의 1차 검토 후 재무팀이 최종 승인한다. "
   "10만원 이상의 비용은 사전 승인이 필요하며 사전 승인 없이 집행된 비용은 정산이 거부될 수 있다. "
   "정산 신청은 지출일로부터 30일 이내에 완료해야 하며 법인카드 사용 내역은 매월 5일까지 제출한다.")),
 dict(title="정보보안 정책", category="보안", department="보안팀", year=2025, body=(
   "외부 USB 등 이동식 저장장치의 사용은 원칙적으로 금지되며 모든 파일 공유는 사내 승인된 드라이브를 통해서만 이루어진다. "
   "외부 협력사와 자료를 공유할 때는 반드시 보안팀의 사전 승인을 받아야 하며 개인 이메일이나 메신저를 통한 전송은 사규 위반이다. "
   "사내 문서에는 보안 등급(대외비·사내한정·공개)을 표기해야 한다. 비밀번호는 90일마다 변경한다.")),
 dict(title="재택근무 및 근태", category="인사", department="인사팀", year=2023, body=(
   "재택근무는 주 2회까지 가능하며 전날 18시까지 소속 팀 채널에 사전 공유해야 한다. "
   "재택 중에도 사내 시스템 접속 시 반드시 VPN을 사용한다. 코어타임은 오전 10시부터 오후 4시까지이며 이 시간에는 메신저 응답을 유지한다. "
   "사전 공유 없이 반복적으로 재택하면 팀장 재량으로 제한될 수 있다.")),
 dict(title="시스템 오류 코드 안내", category="IT", department="IT지원팀", year=2025, body=(
   "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다. "
   "ERR-5010은 네트워크 연결 문제로 VPN 연결 상태를 먼저 확인한다. ERR-6003은 권한 부족으로 시스템 관리자에게 권한을 신청한다. "
   "동일 오류가 반복되면 IT지원팀에 문의해 계정 상태를 점검받는다.")),
 dict(title="신입사원 온보딩", category="인사", department="인사팀", year=2024, body=(
   "입사 첫날 인사팀에서 사내 계정과 장비를 지급하며 보안 서약서 작성이 필수다. "
   "입사 1주일 이내에 정보보안과 성희롱 예방 필수 교육을 이수해야 한다. 수습 기간은 3개월이며 종료 전 팀장과 1차 면담을 진행한다.")),
]

# 제품 매뉴얼 (템플릿으로 다수 생성 → 볼륨 + 필터용 메타데이터)
suppliers = ["파워셀", "볼트원", "에너지코어"]
manuals = []
for n in range(100, 124):                       # 제품 X-100 ~ X-123
    pid = f"X-{n}"
    fw = f"{1 + n % 3}.{n % 5}"
    yr = 2022 + (n % 4)
    manuals.append(dict(
        title=f"제품 {pid} 매뉴얼", category="제품", department="기술지원팀", year=yr, product_id=pid,
        body=(f"제품 {pid}은(는) 소형 산업용 컨트롤러다. 펌웨어 최신 버전은 {fw}이며 설정 메뉴의 업데이트 항목에서 갱신한다. "
              f"배터리는 {suppliers[n % 3]}이(가) 공급하며 정격 전압은 {3 + n % 3}.3V다. "
              f"부팅이 반복되면 펌웨어 {fw}로 재설치하고 그래도 안 되면 기술지원팀에 문의한다. "
              f"방수 등급은 IP{54 + n % 3}이며 작동 온도는 영하 10도에서 영상 {40 + n % 5}도까지다.")))

documents = []
for i, d in enumerate(POLICY + manuals):
    documents.append(dict(doc_id=f"D{i:03d}", **d))

total_chars = sum(len(d["body"]) for d in documents)
print(f"문서 {len(documents)}건, 총 {total_chars:,}자")
print("카테고리:", sorted(set(d["category"] for d in documents)))
print("예시 문서:", documents[0]["title"], "|", documents[0]["category"], documents[0]["year"])

### 3 · 청킹 + 메타데이터 부착
각 문서의 본문을 청킹하고, 모든 청크에 원본 문서의 메타데이터를 물려줍니다.
이 메타데이터가 나중에 필터링의 기준이 됩니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=40)

chunks = []   # 각 청크 = {text, doc_id, title, category, department, year, chunk_idx, (product_id)}
for d in documents:
    parts = splitter.split_text(d["body"])
    for j, p in enumerate(parts):
        meta = {k: d[k] for k in ("doc_id", "title", "category", "department", "year")}
        if "product_id" in d:
            meta["product_id"] = d["product_id"]
        chunks.append({"text": p, "chunk_idx": j, **meta})

print(f"총 청크 {len(chunks)}개 (문서 {len(documents)}건에서)")
print("청크 예시:", {k: chunks[0][k] for k in ("text", "category", "year")})

### 4 · 임베딩 + 벡터DB (메타데이터를 payload로)
청크 텍스트를 임베딩하고, 메타데이터를 함께 payload로 저장합니다. point id는 청크 인덱스와 같게 둡니다.

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
def embed(xs): return embedder.encode(xs, normalize_embeddings=True)

vecs = embed([c["text"] for c in chunks])           # 최초 1회 모델 다운로드로 시간이 걸릴 수 있음
qc = QdrantClient(":memory:")
qc.recreate_collection("kb", vectors_config=VectorParams(size=vecs.shape[1], distance=Distance.COSINE))
qc.upsert("kb", points=[PointStruct(id=i, vector=vecs[i].tolist(), payload=chunks[i]) for i in range(len(chunks))])
print("저장 완료:", qc.count("kb").count, "청크 (payload에 메타데이터 포함)")

## Part 2 · 메타데이터 필터링 검색
### 5 · 필터 없이 vs 필터로
같은 질문이라도 메타데이터로 범위를 좁히면 원하는 문서만 검색됩니다.
Qdrant의 `query_filter`로 카테고리·부서·연도 조건을 건다.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

def search(query, k=5, category=None, department=None, min_year=None):
    conds = []
    if category:   conds.append(FieldCondition(key="category",   match=MatchValue(value=category)))
    if department: conds.append(FieldCondition(key="department", match=MatchValue(value=department)))
    if min_year:   conds.append(FieldCondition(key="year",       range=Range(gte=min_year)))
    flt = Filter(must=conds) if conds else None
    qv = embed([query])[0]
    pts = qc.query_points("kb", query=qv.tolist(), limit=k, query_filter=flt).points
    return [(p.payload["category"], p.payload["year"], p.payload["text"][:34]) for p in pts]

q = "펌웨어는 어떻게 업데이트하나요?"
print("[필터 없음]")
for r in search(q, k=3): print("  ", r)
print("\n[category=제품 · 2024년 이후]")
for r in search(q, k=3, category="제품", min_year=2024): print("  ", r)

### 6 · [실습] 여러 필터 조합
필터를 바꿔가며 결과가 어떻게 달라지는지 확인하세요. 부서·연도·카테고리를 조합해 봅니다.

In [ ]:
for label, kw in [
    ("보안 카테고리만",        dict(category="보안")),
    ("IT지원팀 문서만",        dict(department="IT지원팀")),
    ("2025년 이후만",          dict(min_year=2025)),
]:
    print(f"[{label}]  질문: 인증 토큰이 만료되면?")
    for r in search("인증 토큰이 만료되면?", k=2, **kw): print("  ", r)
    print()
# TODO: 내가 궁금한 조건으로 필터를 조합해 검색해 보세요 (예: category="제품", min_year=2023)

## Part 3 · 검색 기법 비교 + 평가
### 7 · BM25 · 하이브리드 · 리랭커
검색 기법들을 청크 인덱스를 반환하는 함수로 만들고, 뒤에서 점수로 비교합니다.

In [ ]:
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def tok(t): return re.findall(r"[A-Za-z0-9\-]+|[가-힣]+", t)
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def vec_idx(query, n=10):
    qv = embed([query])[0]
    return [p.id for p in qc.query_points("kb", query=qv.tolist(), limit=n).points]
def bm25_idx(query, n=10):
    s = bm25.get_scores(tok(query)); return sorted(range(len(s)), key=lambda i: -s[i])[:n]
def rrf(lists, k=60):
    sc = {}
    for ranks in lists:
        for r, i in enumerate(ranks): sc[i] = sc.get(i, 0) + 1/(k + r + 1)
    return sorted(sc, key=lambda i: -sc[i])
def hybrid_idx(query, n=10):
    return rrf([vec_idx(query, n), bm25_idx(query, n)])[:n]

reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")   # 다국어
def rerank_idx(query, n=3):
    cand = hybrid_idx(query, 12)
    sc = reranker.predict([(query, chunks[i]["text"]) for i in cand])
    return [cand[j] for j in sorted(range(len(cand)), key=lambda j: -sc[j])[:n]]
print("기법 준비: vec / bm25 / hybrid / rerank")

### 8 · 평가셋 + Recall@k · MRR
질문과 정답 키워드 쌍을 만들어 기법별 점수를 냅니다. 정답 키워드가 청크에 있으면 정답으로 칩니다.
> 예시 5개입니다. 내 문서에 맞게 늘리면 비교가 더 또렷해집니다.

In [ ]:
EVALSET = [
    ("연차는 며칠 전에 신청하나요?",        "3영업일"),
    ("비용은 누가 최종 승인해요?",          "재무팀"),
    ("인증 토큰 만료 코드는?",              "ERR-4021"),
    ("외부 USB 써도 되나요?",               "금지"),
    ("재택 근무 코어타임은?",               "10시"),
]

def evaluate(fn, k=3):
    recall, mrr = 0, 0.0
    for q, ans in EVALSET:
        idxs = fn(q, k)
        hit = [r for r, i in enumerate(idxs) if ans in chunks[i]["text"]]
        if hit: recall += 1; mrr += 1/(hit[0]+1)
    n = len(EVALSET); return round(recall/n, 3), round(mrr/n, 3)

print(f"{'기법':<12}{'Recall@3':>10}{'MRR':>8}")
print("-"*30)
for name, fn in [("벡터", vec_idx), ("BM25", bm25_idx), ("하이브리드", hybrid_idx), ("리랭커", rerank_idx)]:
    rec, mrr = evaluate(fn); print(f"{name:<12}{rec:>10}{mrr:>8}")

## Part 4 · 최종 RAG 시스템
### 9 · 필터 + 하이브리드 + 리랭커 + 생성
메타데이터 필터로 범위를 좁히고, 하이브리드와 리랭커로 정밀하게 고른 뒤 근거로 답합니다. 생성에서만 LLM을 씁니다.

In [ ]:
SYSTEM = "아래 [자료]의 내용만 근거로 답하라. 없으면 '문서에 없음'이라고만 답하라. 답 끝에 근거로 삼은 자료의 제목을 (출처: 제목) 형식으로 표기하라."

def filtered_idx(query, n=12, category=None, min_year=None):
    conds = []
    if category: conds.append(FieldCondition(key="category", match=MatchValue(value=category)))
    if min_year: conds.append(FieldCondition(key="year", range=Range(gte=min_year)))
    flt = Filter(must=conds) if conds else None
    qv = embed([query])[0]
    return [p.id for p in qc.query_points("kb", query=qv.tolist(), limit=n, query_filter=flt).points]

def final_rag(query, category=None, min_year=None, k=3):
    cand = filtered_idx(query, 12, category=category, min_year=min_year)   # 필터 + 벡터로 넓게
    if not cand: return "조건에 맞는 문서가 없습니다."
    sc = reranker.predict([(query, chunks[i]["text"]) for i in cand])      # 리랭커로 정밀
    best = [cand[j] for j in sorted(range(len(cand)), key=lambda j: -sc[j])[:k]]
    ctx = "\n".join(f"[{i+1}] ({chunks[d]['title']}) {chunks[d]['text']}" for i, d in enumerate(best))
    out = _c.chat.completions.create(model=LLM_MODEL, temperature=0.2, max_tokens=200,
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":f"[자료]\n{ctx}\n\n[질문] {query}"}])
    return out.choices[0].message.content.split("</think>")[-1].strip()

print("Q: 연차는 며칠 전에 신청하나요?")
print("A:", final_rag("연차는 며칠 전에 신청하나요?"), "\n")
print("Q: X-105 펌웨어 어떻게 올려요?  (category=제품 필터)")
print("A:", final_rag("X-105 펌웨어 어떻게 올려요?", category="제품"), "\n")
print("Q: 사내 헬스장 운영시간은?  (문서에 없음 기대)")
print("A:", final_rag("사내 헬스장 운영시간은?"))

### 10 · [도전] 질문에서 필터를 자동으로 뽑기 (선택)
지금은 필터를 직접 넘겼습니다. LLM에게 질문을 주고 category나 연도 조건을 JSON으로 뽑게 하면, 사용자가 자연어로만 물어도 필터가 걸립니다. 아래를 완성해 보세요.

In [ ]:
import json

def infer_filter(query):
    cats = sorted(set(c["category"] for c in chunks))
    prompt = (f"질문에서 검색 필터를 뽑아 JSON으로만 답하라. category는 {cats} 중 해당되면 하나, 없으면 null. "
              f'형식: {{"category": null}}\n질문: {query}')
    raw = _c.chat.completions.create(model=LLM_MODEL, temperature=0,
        messages=[{"role":"user","content":prompt}]).choices[0].message.content
    raw = raw.split("</think>")[-1].strip().strip("`")
    if raw[:4].lower() == "json": raw = raw[4:].strip()
    try: return json.loads(raw)
    except Exception: return {"category": None}

q = "제품 X-110 방수 등급 알려줘"
f = infer_filter(q)
print("추론된 필터:", f)
print("답:", final_rag(q, category=f.get("category")))
# TODO: 연도 조건(min_year)도 뽑도록 프롬프트를 확장해 보세요

### 11 · [내 문서 적용 + 회고]
- `POLICY`와 `manuals`를 내 실제 문서로 바꾸고(메타데이터 포함), 3~10번을 다시 실행하세요.
- 회고 3줄: ① 메타데이터 필터가 검색 품질을 어떻게 바꿨나 ② 어떤 기법이 가장 좋았나 ③ 어떤 질문에서 실패했나

## 산출물 체크리스트
- [ ] 여러 문서로 지식베이스를 만들고 메타데이터를 붙여 청킹했다
- [ ] 필터 있는/없는 검색 결과 차이를 확인했다
- [ ] 벡터·BM25·하이브리드·리랭커를 점수(Recall@k·MRR)로 비교했다
- [ ] 필터 + 하이브리드 + 리랭커 + 생성으로 최종 시스템을 만들었다
- [ ] 내 문서로 바꿔 재실행하고 회고를 남겼다